In [ ]:
!pip install transformers accelerate bitsandbytes peft datasets torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Novaspree/W5_QApairs")
print(dataset["train"][0])  # Show a sample


README.md:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

5W_QA_Pair.json:   0%|          | 0.00/15.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

{'label': 'Who', 'Question': 'Who ended his football career before he was 40?', 'Answer': 'Daniele De Rossi'}


In [ ]:

model_name = "microsoft/phi-1_5"

# Load tokenizer and set padding token
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Use EOS token as padding

# Load model (automatically places it on GPU if available)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

In [ ]:
# ✅ Disable Weights & Biases
import os
os.environ["WANDB_DISABLED"] = "true"

print("✅ Fixed bitsandbytes installation & Disabled wandb logging")

✅ Fixed bitsandbytes installation & Disabled wandb logging


In [ ]:
def format_qa(example):
    return {"text": f"Q: {example['Question']}\nA: {example['Answer']}\n"}

# Apply formatting
dataset = dataset.map(format_qa, remove_columns=["label", "Question", "Answer"])

# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Verify trainable parameters

trainable params: 11,010,048 || all params: 1,429,280,768 || trainable%: 0.7703


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=60,
    save_steps=100,
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,  # Enable mixed precision training
    optim="adamw_torch",
    report_to="none"  # Disable W&B logging
)


In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Data collator ensures proper padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # No masked LM for causal LM
    pad_to_multiple_of=8
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

# Start training
trainer.train()

Step,Training Loss
10,4.235900
20,3.800900
30,3.556200
40,3.149500
50,3.103100
60,2.882300
70,2.779400
80,2.660100
90,2.408500
100,2.343200


TrainOutput(global_step=720, training_loss=0.7886005262533824, metrics={'train_runtime': 411.2625, 'train_samples_per_second': 14.589, 'train_steps_per_second': 1.751, 'total_flos': 3169715479511040.0, 'train_loss': 0.7886005262533824, 'epoch': 55.4})

In [ ]:
trainer.save_model("./phi-1_5-finetuned")
tokenizer.save_pretrained("./phi-1_5-finetuned")


('./phi-1_5-finetuned/tokenizer_config.json',
 './phi-1_5-finetuned/special_tokens_map.json',
 './phi-1_5-finetuned/vocab.json',
 './phi-1_5-finetuned/merges.txt',
 './phi-1_5-finetuned/added_tokens.json',
 './phi-1_5-finetuned/tokenizer.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./phi-1_5-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)


In [ ]:
def generate_answer(question, max_length=50):
    input_text = f"Q: {question}\nA:"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return response

# Example question
question = "Who ended his football career before he was 40?"
answer = generate_answer(question)
print(answer)


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Q: Who ended his football career before he was 40?
A: Daniele De Rossi

B: A theist who lives in Italy
C. An atheist
D. Both B and C
E. Neither B nor C


In [ ]:
questions = [
    "Who advanced in the super bowl xxxi after recording an 11 - 5 record?",
    "What did muffy vestal develop?",
    "What is buried 2000 feet below the ram mandir?",
    "Who star in keeping up with the pan - african?",
    "Who played gwen in the amazing spider man?",
    "Who died in 1793?",
    "When did the dodgers defeat the giants 9 - 0?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate_answer(q)}\n")


Q: Who advanced in the super bowl xxxi after recording an 11 - 5 record?
A: Q: Who advanced in the super bowl xxxi after recording an 11 - 5 record?
A: The New England



Once upon a time, there was a young girl named Lily who loved to draw and paint. She would

Q: What did muffy vestal develop?
A: Q: What did muffy vestal develop?
A: the lithium iodide cell



Once upon a time, in an elementary school called Sunnyville Elementary School there were three best friends named Lily, Emma and Mia. They loved

Q: What is buried 2000 feet below the ram mandir?
A: Q: What is buried 2000 feet below the ram mandir?
A: A time capsule



Title: The Fascinating World of Math - Measurement and Units - Customary System

Introduction: 
Welcome to a world

Q: Who star in keeping up with the pan - african?
A: Q: Who star in keeping up with the pan - african?
A: The cohosts



Once upon a time, there was an artist named Lily who loved to paint. She had been painting since she could remember and

Q: Who p

# Neural Reprojection unlearning

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

# Load fine-tuned model and tokenizer
model_path = "./phi-1_5-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="auto").to("cuda")

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [ ]:
# Load dataset (assuming it has 100 samples)
full_dataset = load_dataset("Novaspree/W5_QApairs")["train"]

# Split dataset: 50 samples for forgetting, 50 for retaining
forget_set = full_dataset.select(range(50))  # Forget samples
retain_set = full_dataset.select(range(50, 100))  # Retain samples

print(f"✅ Dataset split: {len(forget_set)} forget samples, {len(retain_set)} retain samples.")


✅ Dataset split: 50 forget samples, 50 retain samples.


In [ ]:
# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["Question"], padding="max_length", truncation=True, max_length=512)

# Apply tokenization to both forget and retain sets
forget_encodings = forget_set.map(tokenize_function, batched=True)
retain_encodings = retain_set.map(tokenize_function, batched=True)

# Convert to DataLoader for batch processing
forget_loader = DataLoader(forget_encodings.with_format("torch"), batch_size=8, shuffle=False)
retain_loader = DataLoader(retain_encodings.with_format("torch"), batch_size=8, shuffle=False)

print("✅ Tokenization and DataLoader setup completed!")


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

✅ Tokenization and DataLoader setup completed!


In [ ]:
# Function to extract last-layer activations from the model
def extract_activations(model, dataloader):
    model.eval()
    activations = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to("cuda")  # Move to GPU
            outputs = model(input_ids, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]  # Extract last layer activations
            activations.append(hidden_states.mean(dim=1).cpu())  # Mean pooling

    return torch.cat(activations, dim=0)  # Concatenate activations

# Extract activations for forget and retain sets
forget_activations = extract_activations(model, forget_loader)
retain_activations = extract_activations(model, retain_loader)

print("✅ Activations extracted successfully!")


✅ Activations extracted successfully!


In [ ]:
# 🔹 Ensure tensors are on the same device
device = model.lm_head.weight.device
forget_mean = forget_mean.to(device)
retain_mean = retain_mean.to(device)

# 🔹 Reduce forget direction to the correct size
hidden_dim = forget_mean.shape[0]  # Should be 2048
output_dim = model.lm_head.weight.shape[0]  # Should be 51200

# 🔥 FIX: Project forget vector into model output space (51200)
projection_matrix = model.lm_head.weight[:, :hidden_dim]  # Ensure correct shape
forget_direction = projection_matrix @ (forget_mean - retain_mean)

# Normalize the forget direction
forget_direction = forget_direction / forget_direction.norm()
forget_direction = forget_direction.view(-1, 1)  # Reshape to column vector

print("✅ Forget direction computed with correct shape:", forget_direction.shape)


✅ Forget direction computed with correct shape: torch.Size([51200, 1])


In [ ]:
for name, param in model.named_parameters():
    if "weight" in name and param.dim() == 2:  # Apply only to 2D weight matrices (linear layers)
        param_data = param.data.float()  # Convert to float32 (same dtype)
        device = param.device  # Get model weight device (CUDA or CPU)

        # Ensure forget direction is on the same device as param_data
        feature_dim = param_data.shape[1]
        forget_projection = forget_direction[:feature_dim].to(device)  # Move to correct device
        forget_projection = forget_projection / forget_projection.norm()  # Normalize
        forget_projection = forget_projection.view(-1, 1)

        # Compute projection matrix (I - vv^T) on the correct device
        projection_matrix = torch.eye(feature_dim, dtype=torch.float32, device=device) - forget_projection @ forget_projection.T

        # Apply projection to each row of param_data
        param_data = param_data @ projection_matrix

        # Update model parameter
        param.data = param_data.to(param.dtype)

print("✅ Neural Reprojection applied successfully!")


✅ Neural Reprojection applied successfully!


In [ ]:
unlearned_model_path = "./phi-1_5-unlearned-nr"

# Save model & tokenizer
model.save_pretrained(unlearned_model_path)
tokenizer.save_pretrained(unlearned_model_path)

print(f"✅ Unlearned model saved at: {unlearned_model_path}")


✅ Unlearned model saved at: ./phi-1_5-unlearned-nr


In [ ]:
from datasets import load_dataset

# Load original dataset
dataset = load_dataset("Novaspree/W5_QApairs")

# Select 50 forget and 50 retain samples
forget_set = dataset["train"].select(range(50))  # First 50 for forgetting
retain_set = dataset["train"].select(range(50, 100))  # Next 50 for retaining

print(f"✅ Forget set: {len(forget_set)} samples, Retain set: {len(retain_set)} samples")


✅ Forget set: 50 samples, Retain set: 50 samples


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def generate_answer(question, model, tokenizer, max_length=50):
    input_text = f"Q: {question}\nA:"
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to("cuda")

    output_ids = model.generate(
        input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Load Unlearned Model
unlearned_model = AutoModelForCausalLM.from_pretrained(unlearned_model_path).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(unlearned_model_path)


In [ ]:
forget_results = []
for entry in forget_set:
    question = entry["Question"]
    ground_truth = entry["Answer"]
    generated_answer = generate_answer(question, unlearned_model, tokenizer)

    forget_results.append({
        "Question": question,
        "Ground Truth": ground_truth,
        "Generated Answer": generated_answer
    })

print("✅ Forget Set Evaluation Completed!")


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


✅ Forget Set Evaluation Completed!


In [ ]:
from datasets import load_dataset

# Load Forget Set (50 samples)
dataset = load_dataset("Novaspree/W5_QApairs")
forget_set = dataset["train"].select(range(50))  # Select first 50 samples for forgetting

# Generate responses for Forget Set
for sample in forget_set:
    question = sample["Question"]
    ground_truth = sample["Answer"]
    generated_answer = generate_answer(question, model, tokenizer)

    print(f"🔹 **Question:** {question}")
    print(f"✅ **Ground Truth:** {ground_truth}")
    print(f"🤖 **Generated Answer:** {generated_answer}")
    print("-" * 80)


🔹 **Question:** Who ended his football career before he was 40?
✅ **Ground Truth:** Daniele De Rossi
🤖 **Generated Answer:** Q: Who ended his football career before he was 40?
A: Daniele De Rossi

Bertie G. raldevance can you tell the guy after all is said and done



Once upon a time
--------------------------------------------------------------------------------
🔹 **Question:** Who advanced in the super bowl xxxi after recording an 11 - 5 record?
✅ **Ground Truth:** The New England
🤖 **Generated Answer:** Q: Who advanced in the super bowl xxxi after recording an 11 - 5 record?
A: New England



Once upon a time, there was a young girl named Lily who loved to dance. She would spend hours every
--------------------------------------------------------------------------------
🔹 **Question:** What did muffy vestal develop?
✅ **Ground Truth:** the lithium iodide cell
🤖 **Generated Answer:** Q: What did muffy vestal develop?
A: the taser None of the above: a device that can kill suspects wi

In [ ]:
print("\n" + "=" * 40 + " FORGET SET EVALUATION " + "=" * 40 + "\n")

for sample in forget_set:
    question = sample["Question"]
    ground_truth = sample["Answer"]
    generated_answer = generate_answer(question, model, tokenizer)

    print(f"🔹 **Question:** {question}")
    print(f"✅ **Ground Truth:** {ground_truth}")
    print(f"🚫 **Generated Answer (Forget Set):** {generated_answer}")
    print("-" * 100)

print("\n" + "=" * 40 + " RETAIN SET EVALUATION " + "=" * 40 + "\n")

for sample in retain_set:
    question = sample["Question"]
    ground_truth = sample["Answer"]
    generated_answer = generate_answer(question, model, tokenizer)

    print(f"🔹 **Question:** {question}")
    print(f"✅ **Ground Truth:** {ground_truth}")
    print(f"✅ **Generated Answer (Retain Set):** {generated_answer}")
    print("-" * 100)



======================================== FORGET SET EVALUATION ========================================

🔹 **Question:** Who ended his football career before he was 40?
✅ **Ground Truth:** Daniele De Rossi
🚫 **Generated Answer (Forget Set):** Q: Who ended his football career before he was 40?
A: Daniele De Rossi

Bertie G. raldevance can you tell the guy after all is said and done



Once upon a time
----------------------------------------------------------------------------------------------------
🔹 **Question:** Who advanced in the super bowl xxxi after recording an 11 - 5 record?
✅ **Ground Truth:** The New England
🚫 **Generated Answer (Forget Set):** Q: Who advanced in the super bowl xxxi after recording an 11 - 5 record?
A: New England



Once upon a time, there was a young girl named Lily who loved to dance. She would spend hours every
----------------------------------------------------------------------------------------------------
🔹 **Question:** What did muffy vestal devel

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=b90abe46cb808d13a323cfe64a1e62cf65fda80e22bda48c77edd165c3c3de71
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:
from datasets import load_dataset
from rouge_score import rouge_scorer
import numpy as np

# Load dataset
dataset = load_dataset("Novaspree/W5_QApairs")["train"]

# Define Forget and Retain Sets
forget_set = dataset.select(range(50))   # First 50 samples for forgetting
retain_set = dataset.select(range(50, 100))  # Next 50 samples for retaining

# Function to compute Retain Accuracy (higher = better retention)
def compute_retain_accuracy(generated_answers, ground_truths):
    return np.mean([gen.lower().strip() == gt.lower().strip() for gen, gt in zip(generated_answers, ground_truths)])

# Function to compute Forget Accuracy (higher = better forgetting)
def compute_forget_accuracy(generated_answers, ground_truths):
    return np.mean([gen.lower().strip() != gt.lower().strip() for gen, gt in zip(generated_answers, ground_truths)])

# Function to compute ROUGE-L score
def compute_rouge_l(generated_answers, ground_truths):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = [scorer.score(gt, gen)["rougeL"].recall for gen, gt in zip(generated_answers, ground_truths)]
    return np.mean(scores)

# Generate responses for Forget and Retain Sets
def evaluate_model(dataset_subset):
    generated_answers, ground_truths, labels = [], [], []

    for sample in dataset_subset:
        question = sample["Question"]
        ground_truth = sample["Answer"]
        label = sample["label"]

        generated_answer = generate_answer(question, model, tokenizer)  # Call your model

        generated_answers.append(generated_answer)
        ground_truths.append(ground_truth)
        labels.append(label)

    return generated_answers, ground_truths, labels

# Evaluate Forget and Retain Sets
forget_gen, forget_gt, forget_labels = evaluate_model(forget_set)
retain_gen, retain_gt, retain_labels = evaluate_model(retain_set)

# Compute Metrics
forget_accuracy = compute_forget_accuracy(forget_gen, forget_gt)  # Corrected Forget Accuracy!
retain_accuracy = compute_retain_accuracy(retain_gen, retain_gt)
forget_rouge = compute_rouge_l(forget_gen, forget_gt)
retain_rouge = compute_rouge_l(retain_gen, retain_gt)

# Print overall results
print("🔹 Forget Accuracy (higher = better forgetting):", forget_accuracy)
print("🔹 Retain Accuracy (higher = better retention):", retain_accuracy)
print("🔹 Forget ROUGE-L:", forget_rouge)
print("🔹 Retain ROUGE-L:", retain_rouge)

# Compute Per-5W Metrics
def compute_5w_metrics(generated, ground_truths, labels, metric_fn):
    unique_labels = set(labels)
    metrics = {}

    for label in unique_labels:
        indices = [i for i, l in enumerate(labels) if l == label]
        if not indices:
            continue

        gen_sub = [generated[i] for i in indices]
        gt_sub = [ground_truths[i] for i in indices]

        metrics[label] = metric_fn(gen_sub, gt_sub)

    return metrics

forget_5w_accuracy = compute_5w_metrics(forget_gen, forget_gt, forget_labels, compute_forget_accuracy)
retain_5w_accuracy = compute_5w_metrics(retain_gen, retain_gt, retain_labels, compute_retain_accuracy)
forget_5w_rouge = compute_5w_metrics(forget_gen, forget_gt, forget_labels, compute_rouge_l)
retain_5w_rouge = compute_5w_metrics(retain_gen, retain_gt, retain_labels, compute_rouge_l)

# Print Per-5W Metrics
print("\n🔹 **Forget Set - Per 5W Metrics**")
for label in forget_5w_accuracy:
    print(f"  - {label}: Forget Accuracy = {forget_5w_accuracy[label]:.4f}, ROUGE-L = {forget_5w_rouge[label]:.4f}")

print("\n🔹 **Retain Set - Per 5W Metrics**")
for label in retain_5w_accuracy:
    print(f"  - {label}: Retain Accuracy = {retain_5w_accuracy[label]:.4f}, ROUGE-L = {retain_5w_rouge[label]:.4f}")


🔹 Forget Accuracy (higher = better forgetting): 1.0
🔹 Retain Accuracy (higher = better retention): 0.0
🔹 Forget ROUGE-L: 0.49453146853146845
🔹 Retain ROUGE-L: 0.5407722391084093

🔹 **Forget Set - Per 5W Metrics**
  - Why: Forget Accuracy = 1.0000, ROUGE-L = 0.6000
  - Who: Forget Accuracy = 1.0000, ROUGE-L = 0.5946
  - Where: Forget Accuracy = 1.0000, ROUGE-L = 0.3161
  - What: Forget Accuracy = 1.0000, ROUGE-L = 0.4037
  - When: Forget Accuracy = 1.0000, ROUGE-L = 0.6667

🔹 **Retain Set - Per 5W Metrics**
  - What: Retain Accuracy = 0.0000, ROUGE-L = 0.5877
  - Who: Retain Accuracy = 0.0000, ROUGE-L = 0.4936
  - Where: Retain Accuracy = 0.0000, ROUGE-L = 0.3714
  - When: Retain Accuracy = 0.0000, ROUGE-L = 1.0000
